In order to run the following noteboooks, if you haven't done yet, you need to deploy a model that uses `text-embedding-ada-002` as base model and set the deployment name inside .env file as `AZURE_OPENAI_EMBEDDINGS_ENDPOINT`

In [26]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_KEY'],  # this is also the default, it can be omitted
  api_version = "2023-05-15"
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']

SIMILARITIES_RESULTS_THRESHOLD = 0.7
DATASET_NAME = "../embedding_index_3m.json"

Next, we are going to load the Embedding Index into a Pandas Dataframe. The Embedding Index is stored in a JSON file called `embedding_index_3m.json`. The Embedding Index contains the Embeddings for each of the YouTube transcripts up until late Oct 2023.

In [4]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

Next, we are going to create a function called `get_videos` that will search the Embedding Index for the query. The function will return the top 5 videos that are most similar to the query. The function works as follows:

1. First, a copy of the Embedding Index is created.
2. Next, the Embedding for the query is calculated using the OpenAI Embedding API.
3. Then a new column is created in the Embedding Index called `similarity`. The `similarity` column contains the cosine similarity between the query Embedding and the Embedding for each video segment.
4. Next, the Embedding Index is filtered by the `similarity` column. The Embedding Index is filtered to only include videos that have a cosine similarity greater than or equal to 0.75.
5. Finally, the Embedding Index is sorted by the `similarity` column and the top 5 videos are returned.

In [34]:
def cosine_similarity(a, b):
    return abs(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding
    print(f"query_embeddings for query: {query} is  {query_embeddings}")

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )
    print(video_vectors["similarity"])
    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

This function is very simple, it just prints out the results of the search query.

In [29]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. First, the Embedding Index is loaded into a Pandas Dataframe.
2. Next, the user is prompted to enter a query.
3. Then the `get_videos` function is called to search the Embedding Index for the query.
4. Finally, the `display_results` function is called to display the results to the user.
5. The user is then prompted to enter another query. This process continues until the user enters `exit`.

![](media/notebook_search.png)

You will be prompted to enter a query. Enter a query and press enter. The application will return a list of videos that are relevant to the query. The application will also return a link to the place in the video where the answer to the question is located.

Here are some queries to try out:

- What is Azure Machine Learning?
- How do convolutional neural networks work?
- What is a neural network?
- Can I use Jupyter Notebooks with Azure Machine Learning?
- What is ONNX?

In [38]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from imput
# while True:
#     query = input("Enter a query: ")
#     if query == "exit":
#         break
#     videos = get_videos(query, pd_vectors, 5)
#     display_results(videos, query)
query= "what is Azure machine learning?"
videos = get_videos(query, pd_vectors, 5)
display_results(videos, query)

query_embeddings for query: what is Azure machine learning? is  [-0.01010637916624546, -0.018546229228377342, 0.03878822550177574, -0.008166967891156673, 0.021167844533920288, -0.01271824911236763, 0.015427577309310436, 0.032667871564626694, -0.0011512208729982376, 0.02641107700765133, 0.0012596427695825696, -0.016977157443761826, 0.027794979512691498, -0.003262401558458805, -0.0033574230037629604, 0.03531872481107712, -0.02763904631137848, -0.015349611639976501, -0.01259155385196209, -0.020914454013109207, -0.026157686486840248, -0.00690976157784462, -0.014540711417794228, 0.013137318193912506, -0.008434977382421494, -0.011197906918823719, -0.019657248631119728, -0.011032228358089924, -0.02837972529232502, -0.005189630668610334, 0.02568988874554634, -0.02771701291203499, 0.011383077129721642, 0.02978311851620674, -0.010330531746149063, -0.016460631042718887, -0.007518873084336519, 0.0035352834966033697, -0.009706801734864712, 0.005291961133480072, -0.008917393162846565, -0.02313649281

In [8]:
pd_vectors

,speaker,title,videoId,start,seconds,summary,ada_v2
0,"Seth Juarez, Josh Lovejoy, Sarah Bird",You're Not Solving the Problem You Think You'r...,-tJQm4mSh1s,00:00:00,0,Join Seth Juarez as he discusses ethical conce...,"[0.004357332363724, -0.028409153223037, 0.0111..."
1,"Seth Juarez, Josh Lovejoy, Sarah Bird",You're Not Solving the Problem You Think You'r...,-tJQm4mSh1s,00:03:07,187,"In this video, the speaker discusses the chall...","[-0.0038613036740570003, -0.004626247566193000..."
2,"Seth Juarez, Josh Lovejoy, Sarah Bird",You're Not Solving the Problem You Think You'r...,-tJQm4mSh1s,00:06:13,373,The video discusses the limitations of general...,"[0.00287682027556, -0.012365541420876001, 0.02..."
3,"Seth Juarez, Josh Lovejoy, Sarah Bird",You're Not Solving the Problem You Think You'r...,-tJQm4mSh1s,00:09:21,561,The video discusses the importance of consider...,"[0.015913352370262, 0.000721095071639, 0.02349..."
4,"Seth Juarez, Josh Lovejoy, Sarah Bird",You're Not Solving the Problem You Think You'r...,-tJQm4mSh1s,00:12:24,744,The video discusses the importance of understa...,"[5.447878720588051e-06, -0.011837740428745, 0...."
...,...,...,...,...,...,...,...
1404,"Combining the power of Optimum, OpenVINO™, ONN...","Combining the power of Optimum, OpenVINO™, ONN...",zdDseNfbvIw,00:06:06,366,The video demonstrates how to use OpenVINO Exe...,"[-0.007744120433926, 0.017017424106597002, -0...."
1405,"Combining the power of Optimum, OpenVINO™, ONN...","Combining the power of Optimum, OpenVINO™, ONN...",zdDseNfbvIw,00:09:08,548,The video discusses performance optimization t...,"[-0.010141291655600002, -0.009897269308567, 0...."
1406,"Combining the power of Optimum, OpenVINO™, ONN...","Combining the power of Optimum, OpenVINO™, ONN...",zdDseNfbvIw,00:12:11,731,The video discusses the process of quantizatio...,"[-0.016591534018516003, -0.004845560993999001,..."
1407,"Combining the power of Optimum, OpenVINO™, ONN...","Combining the power of Optimum, OpenVINO™, ONN...",zdDseNfbvIw,00:15:13,913,The video demonstrates how to train and deploy...,"[-0.005989842116832, -0.012303071096539001, -0..."
